In [8]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install numpy pvlib shapely

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 1.3 MB/s eta 0:00:01
   ----------------- ---------------------- 0.8/1.8 MB 1.3 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 1.5 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import pvlib 
from shapely.geometry import polygon

In [12]:
from shapely.geometry import Polygon

In [10]:
def get_sun_vector(latitude, longitude, timestamp):
    # Change 'T' to 'min'
    times = pd.date_range(timestamp, periods=1, freq='min', tz='UTC')
    solpos = pvlib.solarposition.get_solarposition(times, latitude, longitude)

    zenith_rad = np.radians(solpos['apparent_zenith'].iloc[0])
    azimuth_rad = np.radians(solpos['azimuth'].iloc[0])

    S_x = np.sin(zenith_rad) * np.sin(azimuth_rad)
    S_y = np.sin(zenith_rad) * np.cos(azimuth_rad)
    S_z = np.cos(zenith_rad)

    return np.array([S_x, S_y, S_z]), solpos['apparent_zenith'].iloc[0], solpos['azimuth'].iloc[0]

In [5]:
# shadow projection engine

def calculate_panel_shading(panel_origin,panel_width,panel_height,panel_tilt,panel_azimuth,obstracle_vertices,sun_vec):
    # project 3d obstacle vertices onto a 3d panel plane
    # converts 2d panel space,and calculate shaded percentage

    # 1.calculate panel normal vector
    tilt_r=np.radians(panel_tilt)
    azim_r=np.radians(panel_azimuth)

    N=np.array([
        np.sin(tilt_r)*np.sin(azim_r),
        np.sin(tilt_r)*np.cos(azim_r),
        np.cos(tilt_r)
        )]

    # ray direction vector(opposite of sun direction vector)
    d=-sun_vec

    # avoid division by zero if sun is parallel to panel
    if np.abs(np.dot(N,d))<1e6:
        return 0.0,"Sun ray is parallel to panel surface."

    # 2.project obstacle 3d vertices onto 3d panel plane 
    projected_3d_points=[]

    for v_obs in obstacle_vertices:
        t=np.dot(N,(panel_origin-V_obs))/np.dot(N,d)

        if t>0:
            # valid forward projection
            P_int=v_obs+t*d
            projected_3d_points.append(P_int)

        if len(projected_3d_points)<3:
            return 0.0,"no shadow cast on panel plane."

        # 3.define 2d coordinate basis on the panel surface
        
        # u_axis=horizontal direction along panel width
        u_axis=np.array([np.cos(azim_r),-np.sin(azim_r),0])

        # v_axis=direction along panel slant height
        v_axis=np.cross(N,u_axis)

        # 4.transform 3d projected points into 2d panel local coordinates
        shadow_2d_points=[]

        for P in projected_3d_points:
            vec_from_origin=P-panel_origin
            u_coord=np.dot(vec_from_origin,u_axis)
            v_coord=np.dot(vec_from_origin,v_axis)
            shadow_2d_points.append((u_coord,v.coord))


        # 5.define 2d polygons for panel and shadow
        panel_poly=Polygon([
            (0,0),
            (panel_width,0),
            (panel_width,panel_height),
            (0,panel_height)
        ])

        # create convex hull or polygon of projected points
        shadow_poly=Polygon(shadow_2d_points).convex_hull

        # 6.calculate polygon intersection
        if not panel_poly.intersects(shadow_poly):
            return 0.0,"Shadow falls outside panel boundaries."

        intersection=panel_poly.intersection.area
        total_area=panel_poly.area
        shaded_percentage=(shaded_area/total_area)*100.0

        return shaded_percentage,{
            "shaded_srea_m2":shaded_area,
            "sunlight_area_m2":total_area-shaded_area,
            "total_area_m2":total_area
        }

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' on line 11 (830269680.py, line 15)

In [6]:
def calculate_panel_shading(panel_origin, panel_width, panel_height, panel_tilt, panel_azimuth, obstacle_vertices, sun_vec):
    # Converts 3D obstacle vertices onto a 2D panel space and calculates shaded percentage

    # 1. Calculate panel normal vector
    tilt_r = np.radians(panel_tilt)
    azim_r = np.radians(panel_azimuth)

    N = np.array([
        np.sin(tilt_r) * np.sin(azim_r),
        np.sin(tilt_r) * np.cos(azim_r),
        np.cos(tilt_r)
    ])

    # Ray direction vector (opposite of sun direction vector)
    d = -np.array(sun_vec)

    # Avoid division by zero if sun ray is parallel to panel
    if np.abs(np.dot(N, d)) < 1e-6:
        return 0.0, "Sun ray is parallel to panel surface."

    # 2. Project obstacle 3D vertices onto 3D panel plane 
    projected_3d_points = []

    for v_obs in obstacle_vertices:
        t = np.dot(N, (panel_origin - v_obs)) / np.dot(N, d)

        if t > 0:
            # Valid forward projection towards panel plane
            P_int = v_obs + t * d
            projected_3d_points.append(P_int)

    if len(projected_3d_points) < 3:
        return 0.0, "No shadow cast on panel plane."

    # 3. Define 2D coordinate basis on the panel surface
    # u_axis = horizontal direction along panel width
    u_axis = np.array([np.cos(azim_r), -np.sin(azim_r), 0])

    # v_axis = direction along panel slant height
    v_axis = np.cross(N, u_axis)

    # 4. Transform 3D projected points into 2D panel local coordinates
    shadow_2d_points = []

    for P in projected_3d_points:
        vec_from_origin = P - panel_origin
        u_coord = np.dot(vec_from_origin, u_axis)
        v_coord = np.dot(vec_from_origin, v_axis)
        shadow_2d_points.append((u_coord, v_coord))

    # 5. Define 2D polygons for panel and shadow
    panel_poly = Polygon([
        (0, 0),
        (panel_width, 0),
        (panel_width, panel_height),
        (0, panel_height)
    ])

    # Create convex hull of 2D projected points
    shadow_poly = Polygon(shadow_2d_points).convex_hull

    # 6. Calculate polygon intersection
    if not panel_poly.intersects(shadow_poly):
        return 0.0, "Shadow falls outside panel boundaries."

    shaded_area = panel_poly.intersection(shadow_poly).area
    total_area = panel_poly.area
    shaded_percentage = (shaded_area / total_area) * 100.0

    return shaded_percentage, {
        "shaded_area_m2": shaded_area,
        "sunlight_area_m2": total_area - shaded_area,
        "total_area_m2": total_area
    }

In [8]:
# example
if __name__ == "__main__":
    # Location & Time setup (e.g., Pune at 9:00 AM)
    lat, lon = 34.05, -118.25
    timestamp = '2026-06-21 16:00:00' # UTC time
    
    # Sun angles & direction vector
    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°")
    
    # Define Solar Panel parameters
    panel_origin = np.array([0.0, 0.0, 1.0]) # Bottom-left corner (x, y, z)
    panel_w, panel_h = 1.0, 2.0 # 1m wide, 2m tall
    panel_tilt = 30.0 # Tilted 30 degrees
    panel_azimuth = 180.0 # Facing South
    
    # Define Obstacle (e.g., a pole or chimney top corners in 3D space)
    # A 3D box obstacle nearby
    obstacle_vertices = [
        np.array([-0.5, 1.0, 2.5]),
        np.array([0.5, 1.0, 2.5]),
        np.array([0.5, 1.5, 2.5]),
        np.array([-0.5, 1.5, 2.5])
    ]

    # Calculate coverage
    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    print(f"Shaded Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Coverage: {100 - shade_pct:.2f}%")
    print(details)


ValueError: Invalid frequency: T. Failed to parse with error message: ValueError("Invalid frequency: T. Failed to parse with error message: KeyError('T'). Did you mean min?") Did you mean min?

In [13]:
if __name__ == "__main__":
    # Location & Time setup (e.g., Pune at 9:00 AM)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' # UTC time (9:00 AM IST)
    
    # Sun angles & direction vector
    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°")
    
    # Define Solar Panel parameters
    panel_origin = np.array([0.0, 0.0, 1.0]) # Bottom-left corner (x, y, z)
    panel_w, panel_h = 1.0, 2.0 # 1m wide, 2m tall
    panel_tilt = 30.0 # Tilted 30 degrees
    panel_azimuth = 180.0 # Facing South
    
    # Define Obstacle (e.g., a pole or chimney top corners in 3D space)
    # A 3D box obstacle nearby
    obstacle_vertices = [
        np.array([-0.5, 1.0, 2.5]),
        np.array([0.5, 1.0, 2.5]),
        np.array([0.5, 1.5, 2.5]),
        np.array([-0.5, 1.5, 2.5])
    ]

    # Calculate coverage
    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    print(f"Shaded Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Coverage: {100 - shade_pct:.2f}%")
    print(details)

Solar Zenith: 50.43°, Azimuth: 74.49°
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Shadow falls outside panel boundaries.


In [15]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # New Geometries & Distinct Shapes for Testing
    test_scenarios = {
        "1. Circular Water Tank Top (8-Point Octagon Approximation)": [
            np.array([0.0 + 0.6 * np.cos(angle), 1.2 + 0.6 * np.sin(angle), 2.8])
            for angle in np.linspace(0, 2 * np.pi, 8, endpoint=False)
        ],
        "2. Overhead HVAC / Solar Water Heater Unit (Cuboid Box Top)": [
            np.array([-0.8, 0.8, 2.3]),
            np.array([0.2, 0.8, 2.3]),
            np.array([0.2, 1.8, 2.3]),
            np.array([-0.8, 1.8, 2.3])
        ],
        "3. Diamond / Rhombus Shaped Roof Vent": [
            np.array([0.0, 1.0, 2.5]),   # North vertex
            np.array([0.5, 1.5, 2.5]),   # East vertex
            np.array([0.0, 2.0, 2.5]),   # South vertex
            np.array([-0.5, 1.5, 2.5])   # West vertex
        ],
        "4. Slanted Parallelepiped Billboard": [
            np.array([-1.2, 0.8, 1.9]),
            np.array([0.2, 1.2, 2.4]),
            np.array([0.2, 1.7, 2.4]),
            np.array([-1.2, 1.3, 1.9])
        ],
        "5. Complex Star-Shaped Skylight Trim (Non-Convex Perimeter)": [
            np.array([0.0, 1.5, 2.6]),
            np.array([0.2, 1.2, 2.6]),
            np.array([0.6, 1.2, 2.6]),
            np.array([0.3, 0.9, 2.6]),
            np.array([0.4, 0.5, 2.6]),
            np.array([0.0, 0.7, 2.6]),
            np.array([-0.4, 0.5, 2.6]),
            np.array([-0.3, 0.9, 2.6]),
            np.array([-0.6, 1.2, 2.6]),
            np.array([-0.2, 1.2, 2.6])
        ]
    }

    # Run simulations
    for name, obstacle_vertices in test_scenarios.items():
        shade_pct, details = calculate_panel_shading(
            panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
            obstacle_vertices, sun_vec
        )
        
        print(f"=== Scenario: {name} ===")
        print(f"Shaded Coverage: {shade_pct:.2f}%")
        print(f"Sunlight Coverage: {100 - shade_pct:.2f}%")
        print(f"Details: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario: 1. Circular Water Tank Top (8-Point Octagon Approximation) ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 2. Overhead HVAC / Solar Water Heater Unit (Cuboid Box Top) ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 3. Diamond / Rhombus Shaped Roof Vent ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 4. Slanted Parallelepiped Billboard ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 5. Complex Star-Shaped Skylight Trim (Non-Convex Perimeter) ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.



In [14]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Test Scenarios Dictionary
    test_scenarios = {
        "1. Tall Chimney / Pole": [
            np.array([0.2, 1.2, 3.0]),
            np.array([0.5, 1.2, 3.0]),
            np.array([0.5, 1.5, 3.0]),
            np.array([0.2, 1.5, 3.0])
        ],
        "2. Wide Parapet Wall": [
            np.array([-1.5, 1.0, 1.8]),
            np.array([1.5, 1.0, 1.8]),
            np.array([1.5, 1.2, 1.8]),
            np.array([-1.5, 1.2, 1.8])
        ],
        "3. Triangular Roof Gable": [
            np.array([0.0, 1.0, 2.8]),
            np.array([-0.8, 1.0, 1.5]),
            np.array([0.8, 1.0, 1.5])
        ],
        "4. Overhanging Canopy / Tree": [
            np.array([-1.0, 0.5, 2.2]),
            np.array([1.0, 0.5, 2.2]),
            np.array([1.0, 2.5, 2.2]),
            np.array([-1.0, 2.5, 2.2])
        ],
        "5. L-Shaped Rooftop Structure": [
            np.array([-0.5, 1.0, 2.0]),
            np.array([0.8, 1.0, 2.0]),
            np.array([0.8, 1.5, 2.0]),
            np.array([0.0, 1.5, 2.0]),
            np.array([0.0, 2.2, 2.0]),
            np.array([-0.5, 2.2, 2.0])
        ]
    }

    # Run simulations
    for name, obstacle_vertices in test_scenarios.items():
        shade_pct, details = calculate_panel_shading(
            panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
            obstacle_vertices, sun_vec
        )
        
        print(f"=== Scenario: {name} ===")
        print(f"Shaded Coverage: {shade_pct:.2f}%")
        print(f"Sunlight Coverage: {100 - shade_pct:.2f}%")
        print(f"Details: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario: 1. Tall Chimney / Pole ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 2. Wide Parapet Wall ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.

=== Scenario: 3. Triangular Roof Gable ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: No shadow cast on panel plane.

=== Scenario: 4. Overhanging Canopy / Tree ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: No shadow cast on panel plane.

=== Scenario: 5. L-Shaped Rooftop Structure ===
Shaded Coverage: 0.00%
Sunlight Coverage: 100.00%
Details: Shadow falls outside panel boundaries.



In [16]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Test Scenarios: Partial Shading & Low-Irradiance Conditions
    test_scenarios = {
        "1. Off-Center Pole Top (Half Shadow On Panel, Half Outside)": {
            "vertices": [
                np.array([0.5, 1.2, 2.2]),
                np.array([1.5, 1.2, 2.2]),
                np.array([1.5, 1.8, 2.2]),
                np.array([0.5, 1.8, 2.2])
            ],
            "ghi_factor": 1.0  # Clear sunny day (100% full sunlight intensity)
        },
        "2. Same Obstacle on Overcast / Less Sunny Day (Low Irradiance)": {
            "vertices": [
                np.array([0.5, 1.2, 2.2]),
                np.array([1.5, 1.2, 2.2]),
                np.array([1.5, 1.8, 2.2]),
                np.array([0.5, 1.8, 2.2])
            ],
            "ghi_factor": 0.35  # Overcast/cloudy day (only 35% ambient light intensity)
        },
        "3. Diagonal Parapet Edge (Partial Corner Overlap)": {
            "vertices": [
                np.array([-0.8, 0.8, 1.8]),
                np.array([0.3, 0.8, 1.8]),
                np.array([0.3, 1.5, 1.8]),
                np.array([-0.8, 1.5, 1.8])
            ],
            "ghi_factor": 1.0
        }
    }

    # Run simulations
    for name, config in test_scenarios.items():
        obstacle_vertices = config["vertices"]
        ghi_factor = config["ghi_factor"]
        
        shade_pct, details = calculate_panel_shading(
            panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
            obstacle_vertices, sun_vec
        )
        
        # Calculate effective irradiance considering cloudy/less sunny conditions
        effective_unshaded_area = details.get("sunlight_area_m2", 0.0) * ghi_factor
        
        print(f"=== Scenario: {name} ===")
        print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
        print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
        print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
        print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
        print(f"Details: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°



AttributeError: 'str' object has no attribute 'get'

In [17]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Test Scenarios: Partial Shading & Low-Irradiance Conditions
    test_scenarios = {
        "1. Off-Center Pole Top (Half Shadow On Panel, Half Outside)": {
            "vertices": [
                np.array([0.5, 1.2, 2.2]),
                np.array([1.5, 1.2, 2.2]),
                np.array([1.5, 1.8, 2.2]),
                np.array([0.5, 1.8, 2.2])
            ],
            "ghi_factor": 1.0  # Clear sunny day (100% full sunlight intensity)
        },
        "2. Same Obstacle on Overcast / Less Sunny Day (Low Irradiance)": {
            "vertices": [
                np.array([0.5, 1.2, 2.2]),
                np.array([1.5, 1.2, 2.2]),
                np.array([1.5, 1.8, 2.2]),
                np.array([0.5, 1.8, 2.2])
            ],
            "ghi_factor": 0.35  # Overcast/cloudy day (35% ambient light intensity)
        },
        "3. Diagonal Parapet Edge (Partial Corner Overlap)": {
            "vertices": [
                np.array([-0.8, 0.8, 1.8]),
                np.array([0.3, 0.8, 1.8]),
                np.array([0.3, 1.5, 1.8]),
                np.array([-0.8, 1.5, 1.8])
            ],
            "ghi_factor": 1.0
        }
    }

    # Run simulations
    for name, config in test_scenarios.items():
        obstacle_vertices = config["vertices"]
        ghi_factor = config["ghi_factor"]
        
        shade_pct, details = calculate_panel_shading(
            panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
            obstacle_vertices, sun_vec
        )
        
        # Safely extract sunlight area depending on return type
        if isinstance(details, dict):
            sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
        else:
            # If string returned (no shadow), full panel receives sunlight
            sunlight_area = panel_w * panel_h
            
        effective_unshaded_area = sunlight_area * ghi_factor
        
        print(f"=== Scenario: {name} ===")
        print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
        print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
        print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
        print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
        print(f"Details: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario: 1. Off-Center Pole Top (Half Shadow On Panel, Half Outside) ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details: Shadow falls outside panel boundaries.

=== Scenario: 2. Same Obstacle on Overcast / Less Sunny Day (Low Irradiance) ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 35%
Effective Direct Solar Exposure: 0.70 m² equivalent
Details: Shadow falls outside panel boundaries.

=== Scenario: 3. Diagonal Parapet Edge (Partial Corner Overlap) ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details: No shadow cast on panel plane.



In [18]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0

    # Scenario 1: Parallel Sun Rays (Edge-on Sunlight)
    print("=== Scenario 1: ERROR CASE - Sun Rays Parallel to Panel ===")
    obstacle_vertices = [
        np.array([0.0, 1.0, 2.0]),
        np.array([1.0, 1.0, 2.0]),
        np.array([0.5, 1.5, 2.0])
    ]
    panel_tilt = 90.0
    panel_azimuth = 90.0
    ghi_factor = 1.0

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 1: ERROR CASE - Sun Rays Parallel to Panel ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details / Output Message: Shadow falls outside panel boundaries.



In [19]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Scenario 2: Shadow Falls Entirely Outside Panel Boundaries
    print("=== Scenario 2: ERROR CASE - Shadow Cast Distant / Outside Panel ===")
    obstacle_vertices = [
        np.array([10.0, 10.0, 5.0]),
        np.array([11.0, 10.0, 5.0]),
        np.array([10.5, 11.0, 5.0])
    ]
    ghi_factor = 1.0

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 2: ERROR CASE - Shadow Cast Distant / Outside Panel ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details / Output Message: No shadow cast on panel plane.



In [20]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Scenario 3: Degenerate Obstacle (Fewer than 3 projected 2D vertices)
    print("=== Scenario 3: ERROR CASE - Degenerate Obstacle (< 3 Vertices) ===")
    obstacle_vertices = [
        np.array([0.2, 1.2, 2.5]),
        np.array([0.5, 1.2, 2.5])
    ]
    ghi_factor = 1.0

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 3: ERROR CASE - Degenerate Obstacle (< 3 Vertices) ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details / Output Message: No shadow cast on panel plane.



In [21]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Scenario 4: Partial Shadow (Half On / Half Outside Panel)
    print("=== Scenario 4: VALID CASE - Half Shadow On Panel & Half Outside ===")
    obstacle_vertices = [
        np.array([0.5, 1.2, 2.2]),
        np.array([1.5, 1.2, 2.2]),
        np.array([1.5, 1.8, 2.2]),
        np.array([0.5, 1.8, 2.2])
    ]
    ghi_factor = 1.0  # Clear sunny day (100% direct solar irradiance)

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")
    

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 4: VALID CASE - Half Shadow On Panel & Half Outside ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 2.00 m² equivalent
Details / Output Message: Shadow falls outside panel boundaries.



In [22]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Scenario 5: Partial Shadow on Overcast Day (30% Reduced Solar Irradiance)
    print("=== Scenario 5: VALID CASE - Partial Shadow on Low Irradiance / Cloudy Day ===")
    obstacle_vertices = [
        np.array([0.5, 1.2, 2.2]),
        np.array([1.5, 1.2, 2.2]),
        np.array([1.5, 1.8, 2.2]),
        np.array([0.5, 1.8, 2.2])
    ]
    ghi_factor = 0.30  # Low irradiance factor (30% sunlight intensity due to heavy cloud cover)

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 5: VALID CASE - Partial Shadow on Low Irradiance / Cloudy Day ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 30%
Effective Direct Solar Exposure: 0.60 m² equivalent
Details / Output Message: Shadow falls outside panel boundaries.



In [23]:
if __name__ == "__main__":
    # Location & Time: Pune, India at 9:00 AM IST (03:30 AM UTC)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Scenario 6: Full Shadow Coverage (100% Covered Panel)
    print("=== Scenario 6: VALID CASE - Large Overhead Obstacle (100% Total Shadow) ===")
    obstacle_vertices = [
        np.array([-2.0, -1.0, 3.0]),
        np.array([2.0, -1.0, 3.0]),
        np.array([2.0, 3.0, 3.0]),
        np.array([-2.0, 3.0, 3.0])
    ]
    ghi_factor = 1.0  # Clear sunny day

    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Output Message: {details}\n")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Scenario 6: VALID CASE - Large Overhead Obstacle (100% Total Shadow) ===
Shaded Panel Coverage: 19.12%
Sunlight Panel Coverage: 80.88%
Sunlight Intensity Factor: 100%
Effective Direct Solar Exposure: 1.62 m² equivalent
Details / Output Message: {'shaded_area_m2': 0.3823858618617207, 'sunlight_area_m2': 1.6176141381382794, 'total_area_m2': 2.0}



In [24]:
if __name__ == "__main__":
    # Location & Time setup (Pune, India at 9:00 AM IST)
    lat, lon = 18.5204, 73.8567
    timestamp = '2026-06-21 03:30:00' 

    sun_vec, zenith, azimuth = get_sun_vector(lat, lon, timestamp)
    print(f"--- Solar Angles ---")
    print(f"Solar Zenith: {zenith:.2f}°, Azimuth: {azimuth:.2f}°\n")

    # Panel Configuration (1m wide, 2m tall, 30° tilt, South facing)
    panel_origin = np.array([0.0, 0.0, 1.0])
    panel_w, panel_h = 1.0, 2.0
    panel_tilt = 30.0
    panel_azimuth = 180.0

    # Obstacle & Irradiance Configuration
    # Partial obstacle (casts ~half shadow on panel, half outside)
    obstacle_vertices = [
        np.array([0.5, 1.2, 2.2]),
        np.array([1.5, 1.2, 2.2]),
        np.array([1.5, 1.8, 2.2]),
        np.array([0.5, 1.8, 2.2])
    ]
    ghi_factor = 0.40  # 40% ambient light intensity (Overcast/cloudy day)

    # Run calculation
    shade_pct, details = calculate_panel_shading(
        panel_origin, panel_w, panel_h, panel_tilt, panel_azimuth,
        obstacle_vertices, sun_vec
    )

    # Safe evaluation of sunlight area
    if isinstance(details, dict):
        sunlight_area = details.get("sunlight_area_m2", panel_w * panel_h)
    else:
        sunlight_area = panel_w * panel_h

    effective_unshaded_area = sunlight_area * ghi_factor

    print("=== Simulation Output ===")
    print(f"Shaded Panel Coverage: {shade_pct:.2f}%")
    print(f"Sunlight Panel Coverage: {100 - shade_pct:.2f}%")
    print(f"Sunlight Intensity Factor: {ghi_factor * 100:.0f}%")
    print(f"Effective Direct Solar Exposure: {effective_unshaded_area:.2f} m² equivalent")
    print(f"Details / Response: {details}")

--- Solar Angles ---
Solar Zenith: 50.43°, Azimuth: 74.49°

=== Simulation Output ===
Shaded Panel Coverage: 0.00%
Sunlight Panel Coverage: 100.00%
Sunlight Intensity Factor: 40%
Effective Direct Solar Exposure: 0.80 m² equivalent
Details / Response: Shadow falls outside panel boundaries.
